### Imports and Ray Initialization

In [ ]:
import ray
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from agamoo_ray import AGAMOO, Evaluator
from agamoo_ray.players import ClonalSelection
from agamoo_ray.objectives.treproblems import RE33

# Initialize the Ray distributed execution engine
if not ray.is_initialized():
    ray.init()

## Experiment Definition

In [ ]:
def run_experiment(assign_strategy='random'):
    """
    Runs the AGAMOO optimization for the RE33 problem using a specific 
    gene assignment strategy (DVA).
    """
    max_eval = 10000
    model = AGAMOO(max_eval=max_eval, change_iter=10, next_iter=5, 
                  max_front=100, assign_gens=assign_strategy, verbose=False)
    
    # Create the global storage actor for 4 variables and 3 objectives
    storage = model.create_storage(nvars=4, nobjs=3)
    
    # Initialize the RE33 problem instances (Time-Delayed Real-world Engineering)
    objectives = [RE33(num=i, obj=i+1) for i in range(3)]
    
    # Define Clonal Selection hyperparameters
    params = {'nclone': 15, 'mutate_args': [0.45, 0.9, 0.01]}
    
    # Initialize Player and Evaluator Actors
    players = [ClonalSelection.options(num_cpus=1).remote(num=i, npop=20, player_param=params, 
                                     objective=objectives[i], storage_actor=storage) 
               for i in range(3)]
    
    evaluators = [Evaluator.options(num_cpus=0).remote(objectives) for _ in range(2)]
    
    # Register players and start the asynchronous optimization process
    model.init_players(players, evaluators)
    model.start_optimize(tqdm_disable=True)
    
    return model.get_results()